# Diaformer on DDXPlus

This test uses a limited set of patients, all questions and answers will be converted to yes/no to match the style of diaformer, assumes diaformer/diaformer_source contains the diaformer github files

## data preprocessing

### import and split

In [1]:
from pathlib import Path
import ast
import pickle
import pandas as pd
import ast
import json

test = "initial"
trainNo = 2000
validNo = 100
testNo = 100
epochs = 10
batch_size = 8
max_turns = 20
rndseed = 10
run_name = "dummy"

project = Path(r"C:\Users\Hzaab\SDP-Sequential-Diagnosis")

data = project / "data"
source = project / "diaformer" / "Diaformer_source"
prepData = project / "diaformer" / "Data" / run_name
output = project / "diaformer" / "outputs" / run_name


In [2]:
evidences = pd.read_json(data / "release_evidences.json", orient="index")

conditions = pd.read_json(data / "release_conditions.json", orient="index")

print("evidences:", len(evidences), ", conditions:", len(conditions))

evidences: 223 , conditions: 49


In [3]:
columns = ["PATHOLOGY", "EVIDENCES", "INITIAL_EVIDENCE"]

train_df = pd.read_csv(
    data / "release_train_patients" / "release_train_patients.csv",
    usecols=columns
)

validation_df = pd.read_csv(
    data / "release_validate_patients" / "release_validate_patients.csv",
    usecols=columns
)

test_df = pd.read_csv(
    data / "release_test_patients" / "release_test_patients.csv",
    usecols=columns
)

train_df = train_df.sample(n=min(trainNo, len(train_df)), random_state=rndseed)
validation_df = validation_df.sample(n=min(validNo, len(validation_df)), random_state=rndseed)
test_df = test_df.sample(n=min(testNo, len(test_df)), random_state=rndseed)

print("Train:", len(train_df), "Validation:", len(validation_df), "Test:", len(test_df))
train_df.head(3)

Train: 2000 Validation: 100 Test: 100


,PATHOLOGY,EVIDENCES,INITIAL_EVIDENCE
281354,Viral pharyngitis,"['E_41', 'E_48', 'E_49', 'E_53', 'E_54_@_V_161...",E_53
642539,Viral pharyngitis,"['E_41', 'E_45', 'E_48', 'E_49', 'E_53', 'E_54...",E_53
505353,Inguinal hernia,"['E_53', 'E_54_@_V_183', 'E_55_@_V_100', 'E_55...",E_129


### convert data to input for diaformer

for every row, remove the initial evidence from the evidences, then save the patient info in the format that diaformer requires

In [4]:
reformat = []

for frame in [train_df, validation_df, test_df]:
    patients = []

    for index, row in frame.iterrows():

        patient_evidences = ast.literal_eval(row["EVIDENCES"])
        initial = row["INITIAL_EVIDENCE"]

        assert initial in patient_evidences, f"Initial evidence not found: {initial}"

        hidden = [evidence for evidence in patient_evidences if evidence != initial]

        patients.append({
            "disease_tag": row["PATHOLOGY"],
            "goal": {
                "explicit_inform_slots": {initial: True},
                "implicit_inform_slots": {evidence: True for evidence in hidden},
            },
        })

    reformat.append(patients)

train_patients, validation_patients, test_patients = reformat

train_patients[0]

{'disease_tag': 'Viral pharyngitis',
 'goal': {'explicit_inform_slots': {'E_53': True},
  'implicit_inform_slots': {'E_41': True,
   'E_48': True,
   'E_49': True,
   'E_54_@_V_161': True,
   'E_54_@_V_181': True,
   'E_55_@_V_20': True,
   'E_55_@_V_21': True,
   'E_55_@_V_33': True,
   'E_55_@_V_163': True,
   'E_56_@_8': True,
   'E_57_@_V_123': True,
   'E_58_@_4': True,
   'E_59_@_3': True,
   'E_91': True,
   'E_181': True,
   'E_201': True,
   'E_204_@_V_10': True,
   'E_227': True}}}

In [5]:
print("mkdir diaformer\Data\\"+run_name)
print("mkdir diaformer\outputs\\"+run_name)

mkdir diaformer\Data\dummy
mkdir diaformer\outputs\dummy


In [6]:

with (prepData / "goal_set.p").open("wb") as file:
    pickle.dump({"train": train_patients, "test": validation_patients}, file) #naming validation test to match the diaformer code, it will be used as validation

with (prepData / "test_goal_set.p").open("wb") as file:
    pickle.dump({"test": test_patients}, file)


diaformer needs a list of all evidences and diagnoses

In [7]:
all_evidences = []

for index, evidence in evidences.iterrows():
    name = evidence["name"]

    if evidence["data_type"] == "B": #B data type is the True/False evidences
        all_evidences.append(name)

    else: #the other one has multiple values, so diafromer needs a token for each possible value
        for value in evidence["possible-values"]:
            all_evidences.append(f"{name}_@_{value}")

symptoms = sorted(set(all_evidences))
diseases = sorted(conditions["cond-name-eng"].unique())

#diaformer requires these special tokens in this order
special = ["[PAD]", "[PAD2]", "[UNK]", "[SEP]","[CLS]", "[MASK]", "[true]", "[false]"]

with open(prepData / "vocab.txt", "w", encoding="utf-8") as file:
    for token in special + symptoms:
        file.write(token + "\n")

    file.write("\n")

    for disease in diseases:
        file.write(disease + "\n")

print("evidence tokens:", len(symptoms))
print("diseases:", len(diseases))

evidence tokens: 972
diseases: 49


## running diaformer

In [8]:
epochs=1

settings = {
    "run": run_name,
    "seed": rndseed,
    "epochs": epochs,
    "batch_size": batch_size,
    "max_turns": max_turns,
    "learning_rate": 5e-5,
    "warmup_steps": 25,
    "min_probability": 0.0,
    "end_probability": 0.9,
    "train_patients": len(train_patients),
    "validation_patients": len(validation_patients),
    "test_patients": len(test_patients),
}

with open(output / "settings.json", "w", encoding="utf-8") as file:
    json.dump(settings, file, indent=2)
(output / "settings.json").write_text(json.dumps(settings, indent=2))


267

In [9]:
print("""
.venv-diaformer-notebook\\Scripts\\activate.bat
cd diaformer/Diaformer_source
""")


.venv-diaformer-notebook\Scripts\activate.bat
cd diaformer/Diaformer_source



In [10]:
print("python Diaformer.py ^")
print("  --dataset_path ..\\Data\\"+run_name+" ^")
print("  --epochs "+str(epochs)+" --batch_size 8 --seed 10 ^")
print("  --lr 5e-5 --warmup_steps 25 ^")
print("  --start_test 1 --num_workers 0 ^")
print("  --max_turn 20 --min_probability 0.0 --end_probability 0.9 ^")
print("  --train_tokenized_path ..\\outputs\\"+run_name+"\\train.txt ^")
print("  --valid_tokenized_path ..\\outputs\\"+run_name+"\\validation.txt ^")
print("  --log_path ..\\outputs\\"+run_name+"\training.log ^")
print("  --model_output_path ..\\outputs\\"+run_name+"\\model ^")
print("  --result_output_path ..\\outputs\\"+run_name+"\\validation_predictions.json")

python Diaformer.py ^
  --dataset_path ..\Data\dummy ^
  --epochs 1 --batch_size 8 --seed 10 ^
  --lr 5e-5 --warmup_steps 25 ^
  --start_test 1 --num_workers 0 ^
  --max_turn 20 --min_probability 0.0 --end_probability 0.9 ^
  --train_tokenized_path ..\outputs\dummy\train.txt ^
  --valid_tokenized_path ..\outputs\dummy\validation.txt ^
  --log_path ..\outputs\dummy	raining.log ^
  --model_output_path ..\outputs\dummy\model ^
  --result_output_path ..\outputs\dummy\validation_predictions.json


In [11]:
validation_results = pd.read_json(output / "validation_predictions.json")
validation_accuracy = (validation_results["target_disease"] == validation_results["pred_disease"]).mean()
print(f"Validation accuracy: {validation_accuracy:.3f}")


Validation accuracy: 0.070


In [12]:
print("python predict.py ^")
print("  --dataset_path ..\\Data\\"+run_name+" ^")
print("  --goal_set_path ..\\Data\\"+run_name+"\\test_goal_set.p ^")
print("  --pretrained_model ..\\outputs\\"+run_name+"\\model ^")
print("  --max_turn 20 --min_probability 0.0 --end_probability 0.9 ^")
print("  --result_output_path ..\\outputs\\"+run_name+"\\test_predictions.json")

python predict.py ^
  --dataset_path ..\Data\dummy ^
  --goal_set_path ..\Data\dummy\test_goal_set.p ^
  --pretrained_model ..\outputs\dummy\model ^
  --max_turn 20 --min_probability 0.0 --end_probability 0.9 ^
  --result_output_path ..\outputs\dummy\test_predictions.json


In [13]:
test_results = pd.read_json(
    output / "test_predictions.json",
    encoding="cp1252"
)
accuracy = (test_results["target_disease"] == test_results["pred_disease"]).mean()
average_questions = test_results["inquiry_symptom"].map(len).mean()

metrics = {
    "validation_accuracy": float(validation_accuracy),
    "test_accuracy": float(accuracy),
    "average_proposition_questions": float(average_questions),
    "test_patients": len(test_results),
}
(output / "metrics.json").write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))


{
  "validation_accuracy": 0.07,
  "test_accuracy": 0.05,
  "average_proposition_questions": 20.0,
  "test_patients": 100
}
